# 04 — HPO results (visualization)

Loads validation HPO artifacts from `models/nested_hpo/`.

Run training first:

```bash
python scripts/run_nested_hpo_gbm.py
# or
python scripts/run_pipeline.py
```

θ* is chosen on val (`2025-01-07` → `2025-06-24`) only. Test is evaluated in the final walk-forward (notebook 05).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from freight_rates.artifacts import require_hpo_artifacts
from freight_rates.evaluation import format_dual_regime_report
from freight_rates.hpo import select_best_config

ROOT = Path("..").resolve()
hpo = require_hpo_artifacts(ROOT / "models")
val_summary = hpo.val_summary
best_row, best_params, best_label = select_best_config(val_summary)
print("θ* =", best_params)
print(format_dual_regime_report(best_row.to_frame().T))

In [ ]:
cols = [
    "label", "max_depth", "min_samples_leaf", "l2_regularization",
    "mae", "mae_lift", "cold_mae_lift", "beats_lag1_cold", "n", "cold_n",
]
ranked = val_summary[cols].sort_values(["beats_lag1_cold", "mae"], ascending=[False, True])
display(ranked)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(val_summary["mae"], val_summary["cold_mae_lift"], alpha=0.7)
best = val_summary.loc[val_summary["label"] == best_label].iloc[0]
ax.scatter(best["mae"], best["cold_mae_lift"], color="crimson", s=80, label=best_label, zorder=3)
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_xlabel("Overall MAE (val)")
ax.set_ylabel("Cold-start (0-4) MAE lift")
ax.set_title("HPO grid — val window")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
display(hpo.val_folds_best)